In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import subprocess
import re

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Add AF, SVLEN, POS

In [ ]:
# Query the VCF file to extract chr, POS, End, ID, ALT, AF, and SVLEN fields and save them in a BED format
# Using sites only VCF file (AoU_srWGS_SV.v8.sites_only.vcf.gz) for the query. Downloaded from AoU CDRv8 directory 
! bcftools query -f '%CHROM\t%POS\t%INFO/END\t%ID\t%ALT\t%INFO/AF\t%INFO/SVLEN\n' AoU_srWGS_SV.v8.sites_only.vcf.gz > \
    AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF.bed

In [ ]:
# Add cytoband (download from UCSC)

# 1. Sort the files (Required for 'map')
sort -k1,1 -k2,2n cnv_vcf/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF.bed > cnv_vcf/annotation/variants.sorted.bed
zcat cnv_vcf/annotation/ucsc_cytoband/cytoBand.txt.gz | sort -k1,1 -k2,2n > cnv_vcf/annotation/ucsc_cytoband/cytoband.sorted.bed

# 2. Map the cytoband names (column 4 of the cytoband file) to your variants
bedtools map -a cnv_vcf/annotation/variants.sorted.bed -b cnv_vcf/annotation/ucsc_cytoband/cytoband.sorted.bed -c 4 -o collapse > \
cnv_vcf/annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband.tsv

In [ ]:
# Map GENCODE v46 genes to the CNVs

# map genes to the CNVs with gencode v46
gencode_df = pd.read_csv('gencode/gencode.v46.gene_transcript_exon_info.tsv', sep='\t')

gencode_df_unique = gencode_df.drop_duplicates(subset=['Gene_ID'], keep='first')
gencode_df_unique['Gene_ID'].duplicated().sum()

unique_genes = gencode_df_unique['Gene_ID'].unique()
# print(f"Unique genes: {len(unique_genes)}")

gencode_df_unique_subset = gencode_df_unique[['Chromosome', 'Gene_Start', 'Gene_End', 'Gene_ID', 'Gene_Name', 'Gene_Length']]
gencode_df_unique_subset = gencode_df_unique_subset.copy()

gencode_df_unique_subset.loc[:, 'Gene_ID'] = (
    gencode_df_unique_subset['Gene_ID']
    .str.split('.', n=1)
    .str[0]
)

gencode_df_unique_subset.to_csv('gencode/gencode.v46.gene_transcript_exon_info_uniqueGenes_subset.tsv', 
                                sep='\t', index=False)

In [ ]:
# 1. Prepare the gene file: Combine ID and Name into a new format
# We take Chrom (1), Start (2), End (3) and create "ID(Name)" as the 4th column.
awk 'BEGIN{OFS="\t"} NR>1 {print $1, $2, $3, $4"("$5")"}' \
gencode/gencode.v46.gene_transcript_exon_info_uniqueGenes_subset.tsv | \
sort -k1,1 -k2,2n > gencode/gencode.v46.gene_transcript_exon_info_uniqueGenes_subset.sorted.bed

# 2. Ensure your CNV file is sorted
sort -k1,1 -k2,2n annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband.tsv \
> annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband.sorted.bed

# 3. Use bedtools map
# -c 4 now refers to the combined "Gene_ID(Gene_Name)" column
bedtools map -a annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband.sorted.bed \
-b gencode/gencode.v46.gene_transcript_exon_info_uniqueGenes_subset.sorted.bed \
-c 4 -o collapse > annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes.tsv

# 4. Change the comma separator to a semicolon
sed -i 's/,/;/g' annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes.tsv

In [ ]:
# Add column names
master_annotaton_v8 = pd.read_csv('annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes.tsv', 
                                 sep='\t',
                                  header=None,
                                 names=['chr', 'start', 'end', 'ID', 'ALT', 'MAF', 'length_bp', 'cytoband', 'gene'])

master_annotaton_v8.to_csv('annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes.tsv',
                          sep='\t',
                          index=False)

# master_annotaton_v8.head(5)
# master_annotaton_v8.shape

## Add QUAL, FILTER, MAC and gnomAD AF

In [ ]:
! bcftools query -f '%ID\t%ALT\t%QUAL\t%FILTER\t%INFO/AF\t%INFO/AC\t%INFO/gnomAD_v4.1.0_SV_SVID\t%INFO/gnomAD_v4.1.0_SV_AF\t%INFO/gnomAD_v4.1.0_SV_AFR_AF\t%INFO/gnomAD_v4.1.0_SV_AMR_AF\t%INFO/gnomAD_v4.1.0_SV_EAS_AF\t%INFO/gnomAD_v4.1.0_SV_EUR_AF\t%INFO/gnomAD_v4.1.0_SV_MID_AF\t%INFO/gnomAD_v4.1.0_SV_FIN_AF\t%INFO/gnomAD_v4.1.0_SV_ASJ_AF\t%INFO/gnomAD_v4.1.0_SV_RMI_AF\t%INFO/gnomAD_v4.1.0_SV_SAS_AF\t%INFO/gnomAD_v4.1.0_SV_AMI_AF\n' AoU_srWGS_SV.v8.sites_only.vcf.gz > \
    AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_QUAL_AF_AC_gnomAD.bed

In [ ]:
# Add column names
df = pd.read_csv('AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_QUAL_AF_AC_gnomAD.bed',
                sep = '\t',
                header=None,
                names=['ID', 'ALT', 'QUAL', 'FILTER', 'AF', 'AC',
                       'gnomAD_v4.1.0_SV_SVID', 'gnomAD_v4.1.0_SV_AF', 'gnomAD_v4.1.0_SV_AFR_AF', 
                       'gnomAD_v4.1.0_SV_AMR_AF', 'gnomAD_v4.1.0_SV_EAS_AF', 'gnomAD_v4.1.0_SV_EUR_AF', 
                       'gnomAD_v4.1.0_SV_MID_AF', 'gnomAD_v4.1.0_SV_FIN_AF', 'gnomAD_v4.1.0_SV_ASJ_AF', 
                       'gnomAD_v4.1.0_SV_RMI_AF', 'gnomAD_v4.1.0_SV_SAS_AF', 'gnomAD_v4.1.0_SV_AMI_AF '],
                 dtype={'AF': str}
                )

# save tsv
df.to_csv('AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_QUAL_AF_AC_gnomAD.tsv',
         sep='\t',
         index=False)

# df.head()

# Add total CNV and gene overlap

In [ ]:
annotation_df = pd.read_csv('AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes.tsv',
                            sep = '\t')
# annotation_df = annotation_df.drop('Unnamed: 0', axis=1)

In [ ]:
# len(annotation_df)
annotation_df.value_counts('ALT')

In [ ]:
# format annotation file to CNVs with multiple overlapping genes
annotation_df['gene'] = annotation_df['gene'].str.split(';')
# expand rows 
annotation_df = annotation_df.explode('gene')

len(annotation_df)

In [ ]:

# 1. Extract the columns using regex
# Group 1: Everything before '(' -> gene_id
# Group 2: Everything between '(' and ')' -> gene_name
regex_pattern = r'([^(]+)\(([^)]+)\)'
annotation_df[['gene_id', 'gene_name']] = annotation_df['gene'].str.extract(regex_pattern)

# 2. Handle missing values (for variants in gene deserts)
annotation_df[['gene_id', 'gene_name']] = annotation_df[['gene_id', 'gene_name']].fillna('.')

# 3. Drop the original 'gene' column
annotation_df = annotation_df.drop(columns=['gene'])

# 4. Final Row Count Check (to ensure your 1.7M+ rows are intact)
print(f"Total rows: {len(annotation_df)}")
print(annotation_df[['gene_id', 'gene_name']].head())

annotation_df.to_csv('AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes_exploded.tsv',
                    sep='\t',
                    index=False)

In [ ]:
# add genecode genic intervals and calculate the overlapling 
gencode_df = pd.read_csv('gencode/gencode.v46.gene_transcript_exon_info_uniqueGenes_subset.tsv', sep = '\t')

gencode_df.shape

duplicated_genes = gencode_df[gencode_df['Gene_Name'].duplicated(keep=False)]

# Sort by Gene_Name so the duplicates are next to each other
print(duplicated_genes.sort_values(by='Gene_Name'))

In [ ]:
# Load the GFF3
gff = pd.read_csv('./gencode/gencode.v46.annotation.gff3.gz', sep='\t', comment='#', header=None, 
                  names=['chrom', 'source', 'feature', 'Gene_Start', 'Gene_End', 'score', 'strand', 'phase', 'attrs'])

# Filter for genes only
gff_genes = gff[gff['feature'] == 'gene'].copy()

# Extract ID and Name using Regex (much faster than splitting)
gff_genes['Gene_ID'] = gff_genes['attrs'].str.extract(r'gene_id=(ENSG[^;]+)')
gff_genes['Gene_Name'] = gff_genes['attrs'].str.extract(r'gene_name=([^;]+)')

# 2. Calculate Gene Length from Gencode coordinates
# We add 1 because genomic coordinates are 1-based and inclusive
gff_genes['Gene_Length'] = (gff_genes['Gene_End'] - gff_genes['Gene_Start']) + 1

# 3. Create a clean base ID for joining
gff_genes['Gene_ID'] = gff_genes['Gene_ID'].str.split('.').str[0]

# Keep only what we need for the join
ref_df = gff_genes[['Gene_ID', 'Gene_Name', 'Gene_Start', 'Gene_End', 'Gene_Length']]

ref_df.head()
# ref_df.shape

In [ ]:
# left join gene information to annotation table
annotation_genic_df = pd.merge(annotation_df, ref_df[['Gene_ID', 'Gene_Name', 'Gene_Start', 'Gene_End', 'Gene_Length']],
                               left_on = 'gene_id',
                              right_on = 'Gene_ID',
                              how='left').fillna('.')
annotation_genic_df[['Gene_Start', 'Gene_End', 'Gene_Length']] = annotation_genic_df[['Gene_Start', 'Gene_End', 'Gene_Length']].replace({'.':0}).astype(int)

In [ ]:
# 1. Ensure your coordinate columns are numeric
cols = ['start', 'end', 'Gene_Start', 'Gene_End']
annotation_genic_df[cols] = annotation_genic_df[cols].apply(pd.to_numeric)

# 2. Calculate the overlap
# Formula: max(0, min(End1, End2) - max(Start1, Start2))
annotation_genic_df['cnv_gene_overlap_length'] = (
    np.minimum(annotation_genic_df['end'], annotation_genic_df['Gene_End']) - 
    np.maximum(annotation_genic_df['start'], annotation_genic_df['Gene_Start']) + 1
)

# 3. If there is no overlap, the math above results in a negative number.
# We replace negative values with 0.
annotation_genic_df['cnv_gene_overlap_length'] = annotation_genic_df['cnv_gene_overlap_length'].clip(lower=0)

# 4. Optional: Calculate % of gene covered by the CNV
annotation_genic_df['cnv_gene_overlap_Pct'] = (
    (annotation_genic_df['cnv_gene_overlap_length'] / annotation_genic_df['Gene_Length'])
    .fillna(0) * 100
).round(2)

print(annotation_genic_df[['ID', 'Gene_Name', 'cnv_gene_overlap_length', 'cnv_gene_overlap_Pct']].head(10))

In [ ]:
annotation_genic_df[
    (annotation_genic_df['gene_name'] != '.') & 
    (annotation_genic_df['Gene_Name'] == '.')
].shape


# drop unnecessary columns
cols_to_drop = ['Gene_ID', 'Gene_Name', 'Gene_Start', 'Gene_End']
annotation_genic_df.drop(columns=cols_to_drop, inplace=True)

annotation_genic_df.to_csv('cnv_vcf/annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes_exploded_GeneOverlaps.tsv', 
                           sep='\t', index=False)


annotation_genic_df= pd.read_csv('cnv_vcf/annotation/AoU_srWGS_SV.v8.sites_only.vcf_Extracted_ID_POS_SVLEN_AF_cytoband_genes_exploded_GeneOverlaps.tsv', 
                           sep='\t')

# END